In [94]:
import os
from glob import glob
import time
import numpy as np
import pandas as pd
from PIL import Image
import base64
import io
import warnings

import dash
from dash import Dash, html, dcc, Output, Input
import plotly.graph_objects as go
import plotly.express as px
from tqdm.auto import tqdm
import matplotlib
from matplotlib import cm


In [12]:
data_path = "../../../Downloads/archive"
data_stats_files = glob(os.path.join(data_path, 'aggregate/*.csv'))
data_deaths_files = glob(os.path.join(data_path, 'deaths/*.csv'))

df = pd.read_csv(data_stats_files[0]).dropna()
death_df = pd.read_csv(data_deaths_files[0]).dropna()

erangel_death_df = death_df[death_df['map'] == 'ERANGEL'].copy()

distance = np.sqrt(
    (erangel_death_df['killer_position_x'] - erangel_death_df['victim_position_x'])**2 + 
    (erangel_death_df['killer_position_y'] - erangel_death_df['victim_position_y'])**2
)

filtered_df = erangel_death_df[distance <= 100000].copy()

max_time = filtered_df['time'].max()

sampled_df = filtered_df.sample(frac=0.015, random_state=42)

erangel_img_pil = Image.open(os.path.join(data_path, 'erangel.jpg'))
buf = io.BytesIO()
erangel_img_pil.save(buf, format='PNG')
img_base64 = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("utf-8")

In [178]:
# 시간 범위 계산
global_time_min = int(sampled_df['time'].min())
global_time_max = int(sampled_df['time'].max())

# plasma 색상 그라디언트 준비
plasma_rgb_list = [
    f"rgb({int(r*255)}, {int(g*255)}, {int(b*255)})"
    for r, g, b, _ in matplotlib.colormaps.get_cmap("plasma").resampled(20)(np.linspace(0, 1, 20))
]
plasma_gradient = "linear-gradient(to right, " + ", ".join(plasma_rgb_list) + ")"

# 히스토그램 (항상 고정)
hist_fig = px.histogram(
    sampled_df,
    x="time",
    nbins=(global_time_max - 0) // 10,
    title="Death Count over Time (Full Range)",
    labels={"time": "Time", "count": "Deaths"}
)
hist_fig.update_layout(height=200, margin=dict(t=10, b=10, l=40, r=20))

# Dash 앱 레이아웃
app = Dash(__name__)
app.layout = html.Div([
    html.H3("ERANGEL Kill Visualizer"),

    html.Button("그리드 필터 해제", id="clear-grid", n_clicks=0, style={"marginBottom": "10px"}),  # 🔘 버튼
    dcc.Store(id='grid-filter', data=None),  # 🧠 상태 저장소
    dcc.Graph(id='time-histogram', figure=hist_fig),
    
    dcc.RangeSlider(
        id='time-slider',
        min=0,
        max=global_time_max,
        step=10,
        value=[global_time_max-10, global_time_max],
        marks={t: str(t) for t in range(global_time_min, global_time_max+1, 100)},
        tooltip={"placement": "bottom", "always_visible": True}
    ),

    html.Div(style={
        "height": "10px",
        "background": plasma_gradient,
        "marginTop": "-10px",
        "marginBottom": "20px",
        "borderRadius": "4px"
    }),

    dcc.Graph(id='map-graph')
])

In [179]:
@app.callback(
    Output('grid-filter', 'data'),
    Input('clear-grid', 'n_clicks'),
    prevent_initial_call=True
)
def clear_grid_filter(n_clicks):
    print("[DEBUG] 그리드 필터 해제됨")
    return None

@app.callback(
    # Output('map-graph', 'clickData', allow_duplicate=True),  # ✅ allow_duplicate=True로 중복 허용
    Input('map-graph', 'clickData'),
    prevent_initial_call=True
)
def print_click_coordinates(clickData):
    if clickData and 'points' in clickData:
        point = clickData['points'][0]
        x = point.get('x', None)
        y = point.get('y', None)
        print(f"[DEBUG] clickData received: x={x}, y={y}")
    return dash.no_update  # ✅ 아무 것도 바꾸지 않음

@app.callback(
    Output('map-graph', 'figure'),
    Input('time-slider', 'value'),
    Input('map-graph', 'hoverData'),
    Input('grid-filter', 'data')
)
def update_figure(selected_range, hoverData, grid_state):
    try:
        time_start, time_end = selected_range
        df_filtered = sampled_df[
            (sampled_df['time'] >= time_start) &
            (sampled_df['time'] <= time_end)
        ]

        # hover → grid_state 갱신 (grid_state가 없을 때만)
        # if grid_state is None and hoverData and 'points' in hoverData:
        #     hover_point = hoverData['points'][0]
        #     hover_x = hover_point.get('x', None)
        #     hover_y = hover_point.get('y', None)
        #     if hover_x is not None and hover_y is not None and 0 <= hover_x <= 800000 and 0 <= hover_y <= 800000:
        #         grid_x = int(hover_x // 10000) * 10000
        #         grid_y = int(hover_y // 10000) * 10000
        #         grid_state = {'x': grid_x, 'y': grid_y}

        # grid 필터 적용
        apply_grid_filter = False
        grid_x = grid_y = None
        if grid_state:
            grid_x = grid_state['x']
            grid_y = grid_state['y']
            df_filtered = df_filtered[
                (df_filtered['victim_position_x'] >= grid_x) & (df_filtered['victim_position_x'] < grid_x + 10000) &
                (df_filtered['victim_position_y'] >= grid_y) & (df_filtered['victim_position_y'] < grid_y + 10000)
            ]
            apply_grid_filter = True

        if len(df_filtered) == 0:
            return go.Figure()

        global_time_min = int(sampled_df['time'].min())
        global_time_max = int(sampled_df['time'].max())

        bin_min = int(df_filtered['time'].min())
        bin_max = int(df_filtered['time'].max())
        bins = list(range(bin_min, bin_max + 10, 10))

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=pd.errors.SettingWithCopyWarning)
            df_filtered.loc[:, "time_bin"] = pd.cut(df_filtered["time"], bins=bins, include_lowest=True)

        grouped = df_filtered.groupby("time_bin", observed=True)

        def get_color_from_time(time_value):
            normalized = (time_value - global_time_min) / (global_time_max - global_time_min)
            r, g, b, _ = matplotlib.colormaps.get_cmap("plasma")(normalized)
            return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, 0.5)"

        fig = go.Figure()
        for bin_range, group in grouped:
            x_vals = []
            y_vals = []
            for row in group.itertuples():
                x_vals += [row.killer_position_x, row.victim_position_x, None]
                y_vals += [row.killer_position_y, row.victim_position_y, None]
            midpoint = (bin_range.left + bin_range.right) / 2
            color = get_color_from_time(midpoint)
            fig.add_trace(go.Scatter(
                x=x_vals, y=y_vals,
                mode='lines',
                line=dict(color=color, width=1.5),
                hoverinfo='skip',
                showlegend=False
            ))

        # Hover 감지용 trace
        fig.add_trace(go.Scatter(
            x=df_filtered['victim_position_x'],
            y=df_filtered['victim_position_y'],
            mode='markers',
            marker=dict(size=6, color='rgba(0,0,0,0.001)'),
            hoverinfo='x+y',
            showlegend=False,
            name='hover-capture'
        ))

        tick_vals = list(range(0, 800001, 100000))

        layout_config = dict(
            title="Arrows for Time Range: {} ~ {}{}".format(
                time_start, time_end,
                f" | Grid ({grid_x}, {grid_y})" if apply_grid_filter else " (전체 맵)"
            ),
            xaxis=dict(range=[0, 800000], showgrid=True, tickvals=tick_vals),
            yaxis=dict(range=[800000, 0], showgrid=True, tickvals=tick_vals, scaleanchor='x'),
            images=[dict(
                source=img_base64,
                xref="x", yref="y",
                x=0, y=0,
                sizex=800000, sizey=800000,
                sizing="stretch", opacity=0.3,
                layer="below"
            )],
            height=800,
            margin=dict(t=40, b=90, l=10, r=10),
            plot_bgcolor='rgba(0,0,0,0)'
        )

        if apply_grid_filter:
            layout_config["shapes"] = [dict(
                type='rect',
                x0=grid_x,
                x1=grid_x + 10000,
                y0=grid_y,
                y1=grid_y + 10000,
                xref='x', yref='y',
                line=dict(color='gray', width=2),
                fillcolor='rgba(200,200,200,0.2)',
                layer='above'
            )]

        fig.update_layout(**layout_config)
        return fig

    except Exception as e:
        print("콜백 오류:", e)
        return go.Figure()

In [180]:
app.run()

[2025-04-05 23:01:32,321] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 1473, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 882, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 880, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 865, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\dash\dash.py", line 1405, in dispatch
    ctx.run(
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\dash\_callback.py", line 567, in add_context
    raise InvalidC

[DEBUG] clickData received: x=581526.9, y=456842.6


[2025-04-05 23:01:40,482] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 1473, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 882, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 880, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 865, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\dash\dash.py", line 1405, in dispatch
    ctx.run(
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\dash\_callback.py", line 567, in add_context
    raise InvalidC

[DEBUG] clickData received: x=522428.3, y=635800.9


[2025-04-05 23:01:46,291] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 1473, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 882, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 880, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 865, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\dash\dash.py", line 1405, in dispatch
    ctx.run(
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\dash\_callback.py", line 567, in add_context
    raise InvalidC

[DEBUG] clickData received: x=633749.8, y=447155.2


[2025-04-05 23:01:50,337] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 1473, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 882, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 880, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\flask\app.py", line 865, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\dash\dash.py", line 1405, in dispatch
    ctx.run(
  File "C:\Users\darke\.conda\envs\vc_env\lib\site-packages\dash\_callback.py", line 567, in add_context
    raise InvalidC

[DEBUG] clickData received: x=516274.7, y=343410
